In [ ]:
# Load movement CSV data
import pandas as pd
import numpy as np
from pathlib import Path

data_dir = Path('../docs/data/cso')
movement_file = data_dir / 'wcso.csv'

if movement_file.exists():
    df = pd.read_csv(movement_file)
    # Standard cleaning: drop bad rows
    df = df[df['OK'] == 1].copy()
    df = df[df['PRICE'] > 0].copy()
    df['unit_price'] = df['PRICE'] / df['QTY']
    df['log_price'] = np.log(df['unit_price'])
    df['log_move'] = np.log(df['MOVE'].clip(lower=1))
    print(f'Loaded {len(df):,} rows, {df["UPC"].nunique()} UPCs, '
          f'{df["STORE"].nunique()} stores, weeks {df["WEEK"].min()}-{df["WEEK"].max()}')
else:
    print(f'Data not found at {movement_file}. Generating synthetic stub...')
    np.random.seed(42)
    n = 10000
    df = pd.DataFrame({
        'STORE': np.random.randint(1, 20, n),
        'UPC': np.random.choice([f'UPC_{i:03d}' for i in range(25)], n),
        'WEEK': np.random.randint(1, 280, n),
        'MOVE': np.random.poisson(50, n),
    })
    # Simulate prices with store-level cost shocks (for Hausman IV)
    store_cost = {s: np.random.normal(0, 0.1) for s in df['STORE'].unique()}
    df['log_price'] = 0.5 + df['STORE'].map(store_cost) + np.random.normal(0, 0.15, n)
    df['unit_price'] = np.exp(df['log_price'])
    # Demand with true elasticity = -2.5 + confounding
    demand_shock = np.random.normal(0, 0.3, n)
    df['log_move'] = 4.0 - 2.5 * df['log_price'] + demand_shock
    # Add confounding: high demand -> higher prices
    df['log_price'] += 0.3 * demand_shock
    df['unit_price'] = np.exp(df['log_price'])
    print(f'Generated {len(df):,} synthetic rows (true elasticity = -2.5)')

In [ ]:
# Construct Hausman IV
# IV = mean log(price) across OTHER stores for same UPC-week

group_sum = df.groupby(['UPC', 'WEEK'])['log_price'].transform('sum')
group_count = df.groupby(['UPC', 'WEEK'])['log_price'].transform('count')

df['hausman_iv'] = (group_sum - df['log_price']) / (group_count - 1)
# Drop UPC-weeks with only one store
df = df[group_count > 1].copy()

print(f'Hausman IV computed. Shape: {df.shape}')
print(f'IV mean: {df["hausman_iv"].mean():.4f}, std: {df["hausman_iv"].std():.4f}')

In [ ]:
# First-stage regression: log_price ~ hausman_iv + controls
from sklearn.linear_model import LinearRegression

X_first = df[['hausman_iv']].values
y_first = df['log_price'].values

reg_first = LinearRegression().fit(X_first, y_first)
y_pred = reg_first.predict(X_first)
ss_res = np.sum((y_first - y_pred) ** 2)
ss_tot = np.sum((y_first - y_first.mean()) ** 2)
r2 = 1 - ss_res / ss_tot

# F-statistic (single regressor)
n, k = len(y_first), 1
f_stat = (r2 / k) / ((1 - r2) / (n - k - 1))

print(f'First-stage R2: {r2:.4f}')
print(f'First-stage F-stat: {f_stat:.1f}')
print(f'F > 10 (Stock-Yogo threshold): {f_stat > 10}')
if f_stat < 10:
    print('WARNING: Weak instrument. Hausman IV may not be relevant.')

In [ ]:
# DML-PLIV estimation of price elasticity
try:
    import doubleml as dml
    from sklearn.ensemble import RandomForestRegressor

    # Prepare DoubleML data
    dml_data = dml.DoubleMLData.from_arrays(
        x=df[['hausman_iv']].values,  # instrument
        y=df['log_move'].values,       # outcome
        d=df['log_price'].values,      # treatment
        z=df[['hausman_iv']].values,   # instrument (same for PLIV)
    )

    # Partially linear IV model
    learner_g = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
    learner_m = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
    learner_r = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)

    dml_pliv = dml.DoubleMLPLIV(
        dml_data,
        ml_g=learner_g,
        ml_m=learner_m,
        ml_r=learner_r,
        n_folds=5,
    )
    dml_pliv.fit()

    elasticity = dml_pliv.coef[0]
    se = dml_pliv.se[0]
    print(f'DML-PLIV Price Elasticity: {elasticity:.3f} (SE: {se:.3f})')
    print(f'95% CI: [{elasticity - 1.96*se:.3f}, {elasticity + 1.96*se:.3f}]')
    print(f'Expected range for grocery: [-3.0, -2.0]')
except ImportError:
    print('doubleml not installed. Install with: pip install doubleml')
    elasticity = None

In [ ]:
# Visualize elasticity distribution (bootstrap)
import matplotlib.pyplot as plt

# Bootstrap 2SLS for elasticity distribution
n_bootstrap = 200
elasticities_iv = []
elasticities_ols = []

for b in range(n_bootstrap):
    idx = np.random.choice(len(df), len(df), replace=True)
    sample = df.iloc[idx]

    # OLS (biased)
    X_ols = sample[['log_price']].values
    y_ols = sample['log_move'].values
    ols_coef = np.linalg.lstsq(np.c_[np.ones(len(X_ols)), X_ols], y_ols, rcond=None)[0][1]
    elasticities_ols.append(ols_coef)

    # 2SLS
    Z = np.c_[np.ones(len(sample)), sample['hausman_iv'].values]
    X = np.c_[np.ones(len(sample)), sample['log_price'].values]
    y = sample['log_move'].values
    # First stage
    X_hat = Z @ np.linalg.lstsq(Z, X, rcond=None)[0]
    # Second stage
    beta = np.linalg.lstsq(X_hat, y, rcond=None)[0]
    elasticities_iv.append(beta[1])

elasticities_iv = np.array(elasticities_iv)
elasticities_ols = np.array(elasticities_ols)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(elasticities_ols, bins=30, alpha=0.5, label=f'OLS (mean={elasticities_ols.mean():.2f})', color='red')
ax.hist(elasticities_iv, bins=30, alpha=0.5, label=f'2SLS/IV (mean={elasticities_iv.mean():.2f})', color='blue')
ax.axvline(-2.5, color='black', linestyle='--', label='True elasticity (-2.5)')
ax.set_xlabel('Price Elasticity')
ax.set_ylabel('Frequency')
ax.set_title('OLS vs 2SLS Price Elasticity Estimates')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Compare OLS to IV: quantify endogeneity bias
from sklearn.linear_model import LinearRegression

# Naive OLS
ols = LinearRegression().fit(df[['log_price']].values, df['log_move'].values)
ols_elasticity = ols.coef_[0]

# Simple 2SLS
first_stage = LinearRegression().fit(df[['hausman_iv']].values, df['log_price'].values)
price_hat = first_stage.predict(df[['hausman_iv']].values)
second_stage = LinearRegression().fit(price_hat.reshape(-1, 1), df['log_move'].values)
iv_elasticity = second_stage.coef_[0]

print('Endogeneity Bias Analysis')
print('=' * 50)
print(f'OLS elasticity:        {ols_elasticity:>8.3f}  (biased toward zero)')
print(f'2SLS/IV elasticity:    {iv_elasticity:>8.3f}  (consistent)')
print(f'Bias (OLS - IV):       {ols_elasticity - iv_elasticity:>8.3f}')
print(f'Bias direction: OLS attenuated toward zero (simultaniety bias)')
print()
print('The OLS estimate is biased because high demand shocks raise both')
print('quantity and price. The Hausman IV isolates exogenous cost variation')
print('across stores, recovering the causal price elasticity.')